# 🚀 Guia de IA Generatica - Introducción y conceptos básicos

En este notebook aprenderás:
- ✅ Qué es la IA Generativa y cómo funciona
- ✅ Cómo usar la API de Gemini (Google AI - GRATIS)
- ✅ Técnicas de Prompt Engineering
- ✅ Construir aplicaciones reales paso a paso

**⚠️ IMPORTANTE - SEGURIDAD:**
- ❌ NUNCA compartas tus API keys en este notebook
- ❌ NO subas este archivo a GitHub si contiene keys
- ✅ Usa variables de entorno o widgets seguros
- ✅ Revoca keys si se exponen accidentalmente

---
**📋 Requisitos previos:**
- Python 3.8+
- Jupyter Notebook o JupyterLab o VS Code
- Conexión a internet (para APIs)

# 0. SETUP INICIAL

## 📚 Instalar Librerías 


Primero instalamos todas las librerías necesarias. ¡Ejecuta esta celda una sola vez!

In [ ]:
# 📦 Instalación de dependencias
# Esto puede tardar 1-2 minutos la primera vez

import sys
import subprocess
print("🔧 Instalando paquetes necesarios...")
print("=" * 60)

# Lista de paquetes a instalar con sus descripciones
packages = [
    ("google-genai", "🌟 Cliente NUEVO de Google para usar Gemini (GRATIS) - Actualizado 2026"),
    ("requests", "🌐 Para hacer peticiones HTTP a APIs REST"),
    ("ipywidgets", "🎨 Widgets interactivos para inputs, botones y formularios"),
    ("markdown", "📝 Para renderizar y procesar texto en formato Markdown"),
    ("Pillow", "🖼️ Librería para procesamiento de imágenes (PIL)")
]

for package, descripcion in packages:
    try:
        print(f"📥 Instalando {package}...")
        print(f"   ℹ️  {descripcion}")
        # Usar subprocess para evitar problemas con espacios en rutas de Windows
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"✅ {package} instalado correctamente")
    except Exception as e:
        print(f"⚠️ Error instalando {package}: {e}")

print("=" * 60)
print("✨ ¡Instalación completada! Ejecuta la siguiente celda.")

## 📚 Importar Librerías



In [ ]:
# Importaciones estándar
import os
import sys
import json
import time
from datetime import datetime
from typing import List, Dict, Optional

# Librerías para APIs
import requests

# Librerías para visualización
from IPython.display import display, Markdown, HTML, Image
import ipywidgets as widgets
from ipywidgets import Layout, Button, Box, VBox, HBox, Text, Textarea, Password

# Importar Google Gemini
try:
    from google import genai
    GEMINI_AVAILABLE = True
except ImportError:
    GEMINI_AVAILABLE = False
    print("⚠️ Google Gemini no disponible (instala con: pip install google-genai)")

# --------------------------------------------------
# ⚙️ CONFIGURACIÓN SSL PARA PROXIES
# --------------------------------------------------

# 🔒 VERIFICAR_SSL: Controla la verificación de certificados SSL
#
# ✅ True (por defecto): Verifica certificados SSL (recomendado y seguro)
# ⚠️ False: Desactiva verificación SSL (solo para proxies)
#
# 💡 ¿Cuándo cambiar a False?
# Si ves errores como:
#   - "SSLError: certificate verify failed"
#   - "CERTIFICATE_VERIFY_FAILED"
#   - Problemas de conexión detrás de un proxy corporativo
#
# ⚠️ IMPORTANTE: Solo cambia a False en entornos corporativos con proxy
# No desactives SSL en redes públicas o no confiables

VERIFICAR_SSL = False  # Cambia a False si estás detrás de un proxy corporativo

if not VERIFICAR_SSL:
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    print("⚠️ ADVERTENCIA: Verificación SSL desactivada")
    print("   Solo para uso en entornos con proxy")

print("✅ Todas las librerías importadas correctamente")
print(f"📊 Python version: {sys.version.split()[0]}")
print(f"🕐 Fecha actual: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🔒 Verificación SSL: {'Activada ✅' if VERIFICAR_SSL else 'Desactivada ⚠️'}")

## 🔐 Configuración de API Keys (Segura)

Usa estos widgets para introducir tus API keys de forma segura. **¡No se mostrarán en pantalla!** Pero 

### 🆓 Cómo obtener tu API key de Gemini (GRATIS):
- **Gemini**: https://makersuite.google.com/app/apikey (GRATIS hasta 60 req/min)


### 🛡️ **SEGURIDAD - MUY IMPORTANTE:**

- ✅ Si expones una key por error, **revócala inmediatamente** desde la plataforma

#### ¿Dónde se guardan tus API keys?- ✅ Antes de subir a GitHub, ejecuta "Restart Kernel & Clear All Outputs"

- ✅ Se almacenan en **memoria RAM** usando `os.environ` (variables de entorno)

- ✅ Usa SIEMPRE los widgets (lo que haremos ahora)

- ✅ Son **TEMPORALES** - se borran automáticamente al:

  - Cerrar el notebook

  - Reiniciar el kernel de Jupyter- ❌ NO compartas screenshots con keys visibles

  - Cerrar VS Code/Jupyter- ❌ NUNCA hagas `print(os.environ['GEMINI_API_KEY'])` y subas el notebook con el output

- ✅ **NO se guardan** en el archivo `.ipynb`- ❌ NUNCA escribas `api_key = "sk-abc123"` directamente en una celda de código

- ✅ **SEGURO para subir a GitHub** - las keys no están en el archivo#### ⚠️ Riesgos a evitar:


In [ ]:
# 🔐 Widgets seguros para API Keys
print("🔑 Configuración de API Keys")
print("=" * 60)
print("")
print("🛡️ SEGURIDAD:")
print("  - Las keys se guardan SOLO EN MEMORIA (os.environ)")
print("  - NO se guardan en el archivo .ipynb")
print("  - Es SEGURO subir este notebook a GitHub después de usarlo")
print("  - Se borrarán al cerrar el kernel o notebook")
print("")

# Widget para Gemini (GRATIS)
gemini_key_widget = Password(
    placeholder='AIza...',
    description='Gemini:',
    disabled=False,
    layout=Layout(width='500px')
)

# Botón para guardar configuración
save_button = Button(
    description='💾 Guardar Key',
    button_style='success',
    layout=Layout(width='200px')
)

status_output = widgets.Output()

def save_keys(b):
    with status_output:
        status_output.clear_output()
        saved = []
        
        if gemini_key_widget.value and gemini_key_widget.value.startswith('AIza'):
            os.environ['GEMINI_API_KEY'] = gemini_key_widget.value
            saved.append("Gemini")
        
        if saved:
            print(f"✅ Key guardada en memoria para: {', '.join(saved)}")
            print("")
            print("🔒 CONFIRMACIÓN DE SEGURIDAD:")
            print("  ✅ Almacenada en: os.environ (memoria RAM)")
            print("  ✅ NO está en el archivo .ipynb")
            print("  ✅ Se borrará al reiniciar kernel")
            print("  ✅ SEGURO para compartir el notebook")
            print("")
            print("⚠️ IMPORTANTE: Si cierras la sesión, tendrás que volver a introducirla.")
        else:
            print("⚠️ No se detectó una key válida. Verifica que empiece con 'AIza'.")

save_button.on_click(save_keys)

# Mostrar interfaz
display(VBox([
    widgets.HTML("<h3>🔐 Introduce tu API Key de Gemini (GRATIS):</h3>"),
    widgets.HTML("<p style='color: #666;'>Obténla en: <a href='https://makersuite.google.com/app/apikey' target='_blank'>makersuite.google.com/app/apikey</a></p>"),
    gemini_key_widget,
    save_button,
    status_output
]))

print("\n💡 TIP: Puedes continuar con el curso sin key. Aprenderás los conceptos igualmente.")

## ✅ Verificación del Entorno

Vamos a verificar que todo esté configurado correctamente.

In [ ]:
# 🔍 Verificación del entorno

print("🔍 VERIFICACIÓN DEL ENTORNO")
print("=" * 60)

# Verificar Python
print(f"✅ Python: {sys.version.split()[0]}")

# Verificar librerías
libraries = {
    "requests": requests,
    "ipywidgets": widgets,
    "google-generativeai": GEMINI_AVAILABLE
}

for lib_name, lib_status in libraries.items():
    status = "✅" if lib_status else "⚠️"
    print(f"{status} {lib_name}: {'Disponible' if lib_status else 'No disponible (opcional)'}")

# Verificar API Keys
print("\n🔑 API Keys detectadas:")
print("\n🔑 API Key detectada:")
gemini_key = os.getenv('GEMINI_API_KEY', None)
status = "✅" if gemini_key else "⚠️"
masked = f"{gemini_key[:8]}...{gemini_key[-4:]}" if gemini_key else "No configurada"
print(f"{status} Gemini: {masked}")

# Verificar conexión a internet
print("\n🌐 Verificando conexión a internet...")
try:
    response = requests.get("https://www.google.com", timeout=5, verify=VERIFICAR_SSL)
    print("✅ Conexión a internet: OK")
except:
    print("⚠️ Sin conexión a internet (necesaria para APIs)")

print("=" * 60)
print("✨ Verificación completada. ¡Estás listo para empezar!")

---
# 1. INTRODUCCIÓN A IA GENERATIVA y APIs


## 1.1 ¿Qué es IA Generativa?

La **Inteligencia Artificial Generativa** es una rama de la IA que puede **crear contenido nuevo** en lugar de solo analizar o clasificar datos existentes.

### 🎨 Ejemplos de Contenido Generado:

| Tipo | Ejemplo | Herramienta |
|------|---------|-------------|
| 📝 **Texto** | Artículos, código, emails, poesía | ChatGPT, Claude, Gemini |
| 🖼️ **Imágenes** | Fotos realistas, arte digital, logos | Midjourney, Stable Diffusion, Leonardo.ai |
| 🎵 **Música** | Composiciones, melodías, efectos | Suno, MusicLM |
| 🎬 **Video** | Clips, animaciones, deepfakes | Runway, Pika Labs |
| 🗣️ **Voz** | Narración, diálogos, clonación | ElevenLabs, PlayHT |

### 🔄 IA Tradicional vs IA Generativa

```
┌─────────────────────────────────────────────────────────────┐
│                    IA TRADICIONAL                           │
├─────────────────────────────────────────────────────────────┤
│  Input: Imagen de perro                                     │
│  Output: "Es un perro" (Clasificación)                      │
│  Función: ANALIZAR datos existentes                         │
└─────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────┐
│                   IA GENERATIVA                             │
├─────────────────────────────────────────────────────────────┤
│  Input: "Dibuja un perro con sombrero"                     │
│  Output: 🎨 [Imagen nueva de perro con sombrero]           │
│  Función: CREAR contenido nuevo                             │
└─────────────────────────────────────────────────────────────┘
```

### 🧠 ¿Cómo funciona?

Los modelos generativos (como GPT, Gemini) son **redes neuronales** entrenadas con millones de ejemplos. Aprenden **patrones** y los usan para generar contenido nuevo coherente.

**No "piensan"** como humanos, pero son muy buenos **prediciendo** qué palabra/pixel/nota viene después.

## 1.2 ¿Qué son los Modelos de IA Generativa?

### 🤖 Definición Simple

Un **modelo de IA generativa** es un programa de ordenador entrenado con millones (o billones) de ejemplos para **predecir y generar** contenido nuevo.

**Analogía**: Es como un estudiante que leyó millones de libros y ahora puede escribir su propio texto imitando los patrones que aprendió.

---

### 📚 ¿Cómo se Entrenan estos Modelos?

El entrenamiento es un proceso de **3 fases**:

#### 1️⃣ **Pre-entrenamiento** (Foundation Model)
```
📖 Datos masivos → Internet, libros, código, etc.
⏱️ Tiempo: Meses
💰 Coste: Millones de $ (GPUs potentes)
🎯 Objetivo: Aprender patrones generales del lenguaje

Ejemplo: GPT-4 fue entrenado con ~13 billones de palabras
```

#### 2️⃣ **Fine-tuning** (Ajuste fino)
```
🎯 Datos específicos → Instrucciones, diálogos, tareas
⏱️ Tiempo: Días/semanas
💰 Coste: Miles de $
🎯 Objetivo: Especializar el modelo para tareas específicas

Ejemplo: ChatGPT = GPT-4 + fine-tuning en conversaciones
```

#### 3️⃣ **RLHF** (Reinforcement Learning from Human Feedback)
```
👥 Humanos califican respuestas → "buena/mala respuesta"
⏱️ Tiempo: Semanas
🎯 Objetivo: Alinear el modelo con valores humanos

Ejemplo: Enseñar a no generar contenido dañino
```

---

### 🎭 Tipos de Modelos Generativos

| Tipo | Qué genera | Ejemplos | Uso típico |
|------|-----------|----------|------------|
| **LLM** (Large Language Model) | Texto | GPT-5.2, Gemini 3 Pro, Claude 4.5 Opus, **Qwen3 235B**, DeepSeek R2 | Chat, código, análisis |
| **Difusión** | Imágenes | DALL-E 4, Stable Diffusion 4, **Qwen-Image 2.0**, Midjourney v7 | Arte, diseño |
| **Multimodal** | Texto + Imágenes + Video | Gemini 3 Pro, GPT-5.2 Vision, **Qwen-VL-Max**, Claude 4.5 | Análisis multimedia |
| **Audio** | Voz, música | Whisper v4, ElevenLabs v3, Suno AI v4 | Transcripción, síntesis, composición |
| **Video** | Videos | Sora 3.0, Runway Gen-4, Pika 3.0 | Creación y edición de clips |

---

### 🏗️ Arquitecturas Principales

#### 🔷 Transformers (La arquitectura dominante)

Inventada por Google en 2017, es la base de casi todos los modelos modernos.

**Componentes clave**:
- **Tokens**: Unidades básicas (palabras o subpalabras)
- **Embeddings**: Representación numérica de palabras
- **Attention**: Mecanismo para "prestar atención" a partes relevantes
- **Layers**: Capas apiladas que procesan la información

**Tipos de Transformers**:

| Arquitectura | Cómo funciona | Ejemplos | Mejor para |
|--------------|---------------|----------|------------|
| **Encoder-only** | Lee y comprende texto | BERT, RoBERTa | Clasificación, análisis |
| **Decoder-only** | Genera texto palabra por palabra | GPT-5.2, Gemini 3 Pro, **Qwen3**, Llama 4 | Generación, chat |
| **Encoder-Decoder** | Lee input, genera output | T5, BART, Flan-UL2 | Traducción, resumen |

---

### 🌍 Principales Proveedores y sus Modelos

#### 1️⃣ **OpenAI** 🇺🇸
| Modelo | Lanzamiento | Parámetros | Características |
|--------|-------------|------------|-----------------|
| GPT-5.2 | 2025 | **~8T** (no oficial) | Líder en benchmarks complejos, multimodal |
| GPT-5.1 | 2025 | **~6T** (no oficial) | Razonamiento avanzado |
| o1-pro | 2025 | **~2T MoE** (estimado) | Razonamiento extremo |
| DALL-E 4 | 2025 | **~10B difusión** | Imágenes hiperrealistas |

#### 2️⃣ **Google** 🇺🇸
| Modelo | Lanzamiento | Parámetros | Características |
|--------|-------------|------------|-----------------|
| **Gemini 3 Pro** | **2026** | **~4T MoE** | **#1 LMSYS Arena, multimodal líder** |
| **Gemini 3 Flash** | **2026** | **~1.5T MoE** | **⚡ 4M tokens contexto** |
| Gemini 3 Ultra | 2026 | **~12T** (estimado) | Máximo rendimiento |

#### 3️⃣ **Anthropic** 🇺🇸
| Modelo | Lanzamiento | Parámetros | Características |
|--------|-------------|------------|-----------------|
| **Claude 4.5 Opus** | **2026** | **~5T MoE** | **Creatividad máxima** |
| Claude 4 Sonnet | 2025 | **~2.5T MoE** | Equilibrio rendimiento/coste |
| Claude 4 Haiku | 2025 | **~800B MoE** | Ultra-rápido |

#### 4️⃣ **xAI** 🇺🇸
| Modelo | Lanzamiento | Parámetros | Características |
|--------|-------------|------------|-----------------|
| **Grok 4.1** | **2026** | **~3.2T MoE** | **2M tokens, "Thinking" mode** |
| Grok 4 Heavy | 2025 | **~4.8T** | Benchmarks técnicos |

#### 5️⃣ **Meta** 🇺🇸
| Modelo | Lanzamiento | Parámetros | Características |
|--------|-------------|------------|-----------------|
| Llama 4 405B | 2025 | **405B** | ✅ Open source, multimodal |
| Llama 4 70B | 2025 | **70B** | Producción |
| Llama 4 8B | 2025 | **8B** | Consumer GPUs |

#### 6️⃣ **Mistral AI** 🇫🇷
| Modelo | Lanzamiento | Parámetros | Características |
|--------|-------------|------------|-----------------|
| Mistral Large 4 | 2026 | **~600B MoE** | Competidor GPT-5.2 |
| Mixtral 8x44B | 2026 | **352B total (44B activos)** | MoE ultra-eficiente |

#### 7️⃣ **Alibaba (Qwen)** 🇨🇳 
| Modelo | Lanzamiento | Parámetros | Características |
|--------|-------------|------------|-----------------|
| **Qwen3 235B** | **2026** | **235B total (22B activos MoE)** | **Apache 2.0, 100+ idiomas** |
| Qwen3-Coder | 2026 | **480B total (32B activos MoE)** | **Codificación agentica líder** |
| Qwen-VL-Max | 2026 | **72B multimodal** | **Visión multimodal avanzada** |

**💰 Coste**: ✅ **Open source + Alibaba Cloud API**

#### 8️⃣ **DeepSeek** 🇨🇳 
| Modelo | Lanzamiento | Parámetros | Características |
|--------|-------------|------------|-----------------|
| **DeepSeek R2** | **2026** | **671B total (37B activos MoE)** | **Razonamiento matemático #1** |
| DeepSeek-V4 | 2026 | **300B** | **Multilingüe superior** |
| DeepSeek-Coder V3 | 2026 | **236B total (21B activos MoE)** | **Codificación open source líder** |

**💰 Coste**: ✅ **100% Open source (Apache 2.0)**

**📊 Nota técnica**: 
- **MoE** = Mixture of Experts (solo activan fracción de parámetros por inferencia)
- Parámetros de modelos cerrados (**OpenAI/Google/Anthropic**) son **estimaciones basadas en benchmarks y fugas**
- Modelos **open source** tienen parámetros **oficiales confirmados**


---

### 📊 Comparativa: ¿Cuál usar?

| Criterio | Mejor opción | Por qué |
|----------|--------------|---------|
| **💰 Gratis** | **Gemini 3 Flash / Qwen3 / DeepSeek R2** | Límites generosos + open source |
| **🧠 Razonamiento** | **GPT-5.2 / Qwen3 235B / DeepSeek R2** | Líderes GPQA/MATH/AIME |
| **⚡ Velocidad** | **Gemini 3 Flash / Claude 4 Haiku** | Sub-segundo responses |
| **🔓 Open Source** | **Qwen3 / DeepSeek R2 / Llama 4** | Apache 2.0 comercial |
| **🎨 Creatividad** | **Claude 4.5 Opus / Gemini 3 Pro** | Narrativa superior |
| **💻 Código** | **Qwen3-Coder / GPT-5.2 / DeepSeek-Coder** | Líderes SWE-Bench |
| **🌍 Multilingüe** | **Qwen3 235B / DeepSeek R2** | **100+ idiomas nativos** |
| **🌍 Privacidad** | **DeepSeek R2 / Qwen3 local** | 100% on-premise |

---

### 🔢 ¿Qué son los "Parámetros"?

Los **parámetros** son como las "neuronas" del modelo:

- **235B MoE** = Qwen3 (solo 22B activos por inferencia)
- **405B** = Llama 4 (todos activos)
- **5T+** ≈ GPT-5.2

**➕ China domina open source**: Qwen3/DeepSeek superan GPT-4 en precio/rendimiento

---

### 💡 Concepto Clave: "Contexto" o "Context Window"

| Modelo | Contexto | Equivalente |
|--------|----------|-------------|
| **Qwen3 235B** | **128K nativo (1M+ extendido)** | **~1M palabras** |
| GPT-5.2 | 2M tokens | ~1.5M palabras |
| **Gemini 3 Pro** | **4M tokens** | **~3M palabras** 🤯 |
| **DeepSeek R2** | **256K+ tokens** | ~200K+ palabras |

---

### 🎯 En este curso usaremos:

✅ **Google Gemini 3 Flash**  + **Google Gemini 2.5 Flash Lite** 

| Característica | **Google Gemini 3 Flash** | **Google Gemini 2.5 Flash Lite** |
| :--- | :--- | :--- |
| **Enfoque Principal** | Razonamiento Crítico / Agentes | Velocidad Extrema / Volumen |
| **Cuota Gratuita (Día)** | **20 Requests** | **500 Requests** |
| **Nivel de Inteligencia** | **Muy Alto** (Capacidad de "Pensar") | **Medio** (Optimizado para latencia) |
| **Ventana de Contexto** | 1 Millón de tokens | 1 Millón de tokens |
| **Programación (Coding)** | Nivel Experto (Arquitectura) | Nivel Funcional (Scripts simples) |
| **Latencia** | Baja | **Ultra-baja** (Casi instantánea) |
| **Multimodalidad** | Texto, Audio, Video e Imagen | Texto, Audio, Video e Imagen |



#### 🧠 ¿En que se diferencian?

##### 🚀 **Google Gemini 3 Flash**
Es el modelo ideal cuando la **precisión** es más importante que la cantidad. 
* **Uso recomendado:** Resolución de problemas matemáticos, debugging de código complejo, análisis profundo de documentos extensos y tareas que requieren lógica de varios pasos.
* **Limitación:** Su cuota de 20 mensajes al día lo hace exclusivo para tareas de "alta importancia".

##### ⚡ **Google Gemini 2.5 Flash Lite**
Es el "caballo de batalla" para el día a día y la **automatización**.
* **Uso recomendado:** Clasificación masiva de correos, resúmenes rápidos de reuniones, traducción de textos, chatbots de respuesta inmediata y tareas repetitivas.
* **Ventaja:** Con 500 peticiones diarias, es prácticamente imposible agotar el límite en un uso personal normal.

---



## 1.3 Cómo Interactuamos con IA Generativa

Hay 3 formas principales de usar modelos de IA:

### 🎮 1. Playgrounds (Interfaz Web)
- **Ejemplos**: ChatGPT.com, Gemini.google.com, Claude.ai
- **Pros**: Fácil, sin código, visual
- **Cons**: Limitado, no automatizable

### 🔌 2. APIs (Programático) ← **¡LO QUE USAREMOS HOY!**
- **Ejemplos**: OpenAI API, Google AI API, Anthropic API
- **Pros**: Automatizable, integrable, potente
- **Cons**: Requiere código, suele ser de pago

### 💻 3. Local (Tu ordenador)
- **Ejemplos**: Ollama, LM Studio, GPT4All
- **Pros**: Privado, gratis, sin internet
- **Cons**: Requiere hardware potente, modelos más pequeños

---

### 📊 Diagrama de Flujo: Cómo funciona una API

```
┌─────────────────────┐
│  👨‍💻 Tu Código      │
│     Python          │
└──────────┬──────────┘
           │
           │ 1. HTTP Request (POST)
           │    {"prompt": "Hola IA"}
           ▼
┌─────────────────────┐
│   🌐 Internet       │
│   (HTTP/HTTPS)      │
└──────────┬──────────┘
           │
           │ 2. Envía a servidor
           ▼
┌─────────────────────┐
│  ☁️ Servidor IA     │
│  OpenAI / Gemini    │
└──────────┬──────────┘
           │
           │ 3. Procesa con GPU
           ▼
┌─────────────────────┐
│  🤖 Modelo LLM      │
│  GPT-4 / Gemini     │
│  (Genera texto)     │
└──────────┬──────────┘
           │
           │ 4. Respuesta generada
           ▼
┌─────────────────────┐
│  ☁️ Servidor IA     │
│  (Envía respuesta)  │
└──────────┬──────────┘
           │
           │ 5. HTTP Response (JSON)
           │    {"text": "¡Hola! ¿Cómo..."}
           ▼
┌─────────────────────┐
│   🌐 Internet       │
└──────────┬──────────┘
           │
           │ 6. Retorna datos
           ▼
┌─────────────────────┐
│  👨‍💻 Tu Código      │
│  (Procesa y muestra)│
└──────────┬──────────┘
           │
           ▼
┌─────────────────────┐
│ 📺 Tu Aplicación    │
│ ¡Hola! ¿Cómo...     │
└─────────────────────┘
```

**🔄 Flujo resumido:**
1. **Tu código** → Envía prompt (texto)
2. **Internet** → Transporta la petición
3. **Servidor IA** → Recibe y procesa
4. **Modelo LLM** → Genera la respuesta
5. **Servidor IA** → Devuelve resultado
6. **Tu código** → Muestra al usuario

💡 **Clave**: Cada llamada es independiente. El modelo NO recuerda conversaciones previas (veremos cómo simular memoria más adelante).

## 1.4 Prompts y Tipos de Técnicas

Un **prompt** es la **instrucción** que le das al modelo de IA. La calidad del prompt determina la calidad de la respuesta.

### 🎯 Técnicas Principales:

#### 1️⃣ **Zero-Shot** (Sin ejemplos)
Le pides directamente lo que quieres, sin dar ejemplos previos.

```
Prompt: "Traduce al inglés: Hola, ¿cómo estás?"
Respuesta: "Hello, how are you?"
```

#### 2️⃣ **Few-Shot** (Con ejemplos)
Le das algunos ejemplos antes de tu pregunta real.

```
Prompt: 
"Clasifica el sentimiento (positivo/negativo/neutral):
- 'Me encanta esta película' → Positivo
- 'Es aburrido' → Negativo  
- 'Es una mesa' → Neutral
- 'Qué día tan horrible' → ?"

Respuesta: "Negativo"
```

#### 3️⃣ **Chain-of-Thought (CoT)** - Razonamiento paso a paso
Pides al modelo que explique su razonamiento.

```
Prompt: "Si una camisa cuesta 20€ y hay 30% descuento, 
         ¿cuánto pagas? Piensa paso a paso."

Respuesta: 
"Paso 1: Calcular descuento: 20 × 0.30 = 6€
 Paso 2: Restar del precio: 20 - 6 = 14€
 Respuesta: 14€"
```

---

### 🎭 Framework RACE para Prompts Efectivos

| Componente | Descripción | Ejemplo |
|------------|-------------|---------|
| **R**ol | Quién quieres que sea el modelo | "Eres un profesor de física" |
| **A**cción | Qué debe hacer | "Explica la gravedad" |
| **C**ontexto | Información adicional relevante | "Para un estudiante de 12 años" |
| **E**jemplo | Formato de respuesta deseado (puede ser desde un texto normal hasta un JSON estructurado) | "Usa analogías simples" |

**Ejemplo completo**:
```
Eres un chef profesional (ROL). 
Dame una receta de paella (ACCIÓN) 
para 4 personas con ingredientes fáciles de conseguir (CONTEXTO).
Usa este formato: Ingredientes, Pasos, Tiempo (EJEMPLO).
```

---

### 💬 Tipos de Mensajes en Conversaciones

Las APIs de chat usan 3 tipos de mensajes en cada interacción:

| Tipo | Propósito | Ejemplo |
|------|-----------|---------|
| **system** | Instrucciones globales, define personalidad y requisitos generales | "Eres un asistente útil y conciso" |
| **user** | Mensaje del usuario | "¿Cuál es la capital de Francia?" |
| **assistant** | Respuesta del modelo IA | "La capital de Francia es París" |

**Estructura típica**:
```python
[
    {"role": "system", "content": "Eres un experto en historia"},
    {"role": "user", "content": "¿Cuándo cayó el Imperio Romano?"},
    {"role": "assistant", "content": "En el año 476 d.C."}, -- RESPUESTA
]
```

El modelo usa el conjunto de mensajes para entender el contexto.

## 1.5 ¿Qué es una API?

**API** = **A**pplication **P**rogramming **I**nterface

Es como un **camarero en un restaurante**:
- 🍽️ Tú (cliente) pides comida (datos/texto)
- 🚶 El camarero (API) lleva tu pedido a la cocina (servidor)
- 👨‍🍳 La cocina (modelo IA) prepara el plato (genera respuesta)
- 🚶 El camarero te trae la comida (respuesta)

### 🔍 Ejemplo REAL: API del Tiempo (Uso Cotidiano)

Vamos a usar **Open-Meteo**, una API del tiempo **GRATIS** y sin autenticación. ¡Algo que todos usamos en el día a día!

**¿Por qué una API del tiempo?**
- 🌤️ Todos consultamos el tiempo regularmente
- 🆓 Completamente gratis, sin límites razonables
- 🔓 No requiere API key ni registro
- 📱 Igual que las apps de tiempo de tu móvil

### 🔍 PASO 1: Entender las URLs de las APIs

---

Antes de hacer nuestra primera petición, necesitas entender cómo se construye una **URL de API**.

#### 📐 Anatomía de una URL de API

Una URL de API tiene varias partes importantes:

```
https://api.open-meteo.com/v1/forecast?latitude=40.4168&longitude=-3.7038&current=temperature_2m
│      │                     │         │                                │
│      │                     │         │                                └─ Parámetros (datos que enviamos)
│      │                     │         └───────────────────────────────── El símbolo ? separa la ruta de los parámetros
│      │                     └─────────────────────────────────────────── Ruta (qué acción queremos)
│      └───────────────────────────────────────────────────────────────── Dominio (servidor que tiene los datos)
└──────────────────────────────────────────────────────────────────────── Protocolo (https = seguro)
```

#### 🧩 Los Parámetros (Query Parameters)

Los parámetros van después del `?` y se separan con `&`:

| Parámetro | Valor | Qué significa |
|-----------|-------|---------------|
| `latitude` | `40.4168` | Latitud de Madrid (coordenada Norte-Sur) |
| `longitude` | `-3.7038` | Longitud de Madrid (coordenada Este-Oeste) |
| `current` | `temperature_2m,wind_speed_10m,weather_code` | Qué datos queremos (temperatura, viento, clima) |
| `timezone` | `Europe/Madrid` | Zona horaria para las fechas |

**💡 Piensa en los parámetros como "rellenar un formulario"** - le dices a la API exactamente qué información necesitas.


### 🔍 PASO 2: Preparar la petición a la API

In [ ]:
# Vamos a consultar el tiempo en Madrid usando Open-Meteo
# Esta API es GRATIS y NO necesita registro

# Paso 2.1: Definir el servidor y la ruta
servidor = "https://api.open-meteo.com"
ruta = "/v1/forecast"

# Paso 2.2: Definir los parámetros (coordenadas de Madrid)
latitud = "40.4168"   # Madrid está a 40.4168° Norte
longitud = "-3.7038"  # Madrid está a 3.7038° Oeste (negativo = Oeste)

# Paso 2.3: Definir qué datos queremos
datos_solicitados = "temperature_2m,wind_speed_10m,weather_code"
# temperature_2m = temperatura a 2 metros del suelo (°C)
# wind_speed_10m = velocidad del viento a 10 metros (km/h)
# weather_code = código del tiempo (0=despejado, 3=nublado, 61=lluvia, etc.)

zona_horaria = "Europe/Madrid"

# Paso 2.4: CONSTRUIR LA URL COMPLETA manualmente (para que veas cómo se hace)
api_url = f"{servidor}{ruta}?latitude={latitud}&longitude={longitud}&current={datos_solicitados}&timezone={zona_horaria}"

print("\n📋 URL construida:")
print(f"   {api_url}\n")

print("🔍 Desglosando la URL:")
print(f"   🌐 Servidor: {servidor}")
print(f"   📂 Ruta: {ruta}")
print(f"   ❓ Parámetros:")
print(f"      - latitude={latitud}")
print(f"      - longitude={longitud}")
print(f"      - current={datos_solicitados}")
print(f"      - timezone={zona_horaria}")

# --------------------------------------------------
# PREPARAR DATOS DE EJEMPLO (por si falla la conexión)
# --------------------------------------------------

# Si no hay internet o hay problemas de red, usaremos estos datos
# Así puedes seguir aprendiendo aunque no funcione la API
datos_ejemplo = {
    "latitude": 40.42,
    "longitude": -3.7,
    "timezone": "Europe/Madrid",
    "current": {
        "time": "2026-03-05T15:00",
        "temperature_2m": 18.5,
        "wind_speed_10m": 12.3,
        "weather_code": 2
    }
}

print("\n✅ URL preparada y lista para usar")
print("🚀 En la siguiente celda haremos la petición HTTP...")

### 🔍 PASO 3: Hacer la petición a la API

In [ ]:
try:
    # --------------------------------------------------
    # ENVIAR LA PETICIÓN HTTP
    # --------------------------------------------------
    
    # requests.get() = solicitar datos al servidor
    # timeout=10 = esperar máximo 10 segundos
    # verify=VERIFICAR_SSL = controla verificación de certificados SSL
    print("⏳ Conectando con el servidor meteorológico...")
    print("   (Enviando petición HTTP GET...)")
    
    response = requests.get(api_url, timeout=10, verify=VERIFICAR_SSL)
    # 'response' contiene toda la respuesta del servidor
    
    # --------------------------------------------------
    # VERIFICAR SI FUNCIONÓ
    # --------------------------------------------------
    
    # Los servidores responden con "códigos de estado":
    # 200 = OK (todo bien)
    # 404 = No encontrado
    # 500 = Error del servidor
    # etc.
    
    if response.status_code == 200:
        print("✅ ¡Petición exitosa!")
        print(f"📊 Código de estado HTTP: {response.status_code} (OK)")
        print(f"⏱️ Tiempo de respuesta: {response.elapsed.total_seconds():.2f} segundos")
        
        # --------------------------------------------------
        # EXTRAER LOS DATOS (parsear JSON)
        # --------------------------------------------------
        
        print("\n📦 Convirtiendo la respuesta JSON a diccionario Python...")
        
        # .json() convierte el texto JSON en un diccionario Python
        data = response.json()
        
        print("✅ Datos convertidos correctamente\n")
        print("📋 JSON completo recibido:")
        print("-" * 60)
        
        # Mostrar el JSON completo de forma interactiva
        from IPython.display import JSON
        display(JSON(data))
        
    else:
        print(f"❌ Error: Código {response.status_code}")
        print("   (La API no respondió correctamente)")
        data = datos_ejemplo
        from IPython.display import JSON
        display(JSON(data))

except Exception as e:
    # Si hay algún error de conexión, usar datos de ejemplo
    print(f"❌ Error de conexión: {type(e).__name__}")
    print(f"   Detalles: {e}")
    print("\n💡 Usando datos de EJEMPLO para continuar el aprendizaje:")
    print("-" * 60)
    data = datos_ejemplo
    from IPython.display import JSON
    display(JSON(data))
    print("\n⚠️ Estos son datos simulados, pero el formato es el mismo")

print("\n" + "=" * 60)
print("💡 En la siguiente celda interpretaremos estos datos...")

### 🔍 PASO 4: Interpretar los datos recibidos (darle formato "friendly")

In [ ]:
# El JSON tiene esta estructura:
# {
#   "current": {
#     "temperature_2m": 18.5,
#     "wind_speed_10m": 12.3,
#     "weather_code": 2
#   }
# }

# Para acceder a un valor: diccionario['clave']['subclave']

temperatura = data['current']['temperature_2m']
viento = data['current']['wind_speed_10m']
codigo_tiempo = data['current']['weather_code']

print(f"\n✅ Datos extraídos del JSON:")
print(f"   temperatura = {temperatura}")
print(f"   viento = {viento}")
print(f"   codigo_tiempo = {codigo_tiempo}")

# --------------------------------------------------
# TRADUCIR EL CÓDIGO DEL TIEMPO
# --------------------------------------------------

# La API devuelve un número (código), nosotros lo convertimos a texto

print("\n🔄 Traduciendo el código del tiempo a descripción legible...")

descripciones_tiempo = {
    0: "☀️ Despejado",
    1: "🌤️ Mayormente despejado",
    2: "⛅ Parcialmente nublado",
    3: "☁️ Nublado",
    45: "🌫️ Niebla",
    48: "🌫️ Niebla con escarcha",
    51: "🌦️ Llovizna ligera",
    61: "🌧️ Lluvia ligera",
    63: "🌧️ Lluvia moderada",
    65: "🌧️ Lluvia intensa",
    80: "🌦️ Chubascos ligeros",
    95: "⛈️ Tormenta"
}

# .get() busca el código en el diccionario
# Si no existe, devuelve un texto por defecto
descripcion = descripciones_tiempo.get(codigo_tiempo, f"Código {codigo_tiempo}")

# --------------------------------------------------
# MOSTRAR RESULTADO FINAL
# --------------------------------------------------

print("\n" + "=" * 60)
print("🌤️ RESULTADO FINAL - TIEMPO EN MADRID:")
print("=" * 60)

print(f"🌡️ Temperatura: {temperatura}°C")
print(f"💨 Viento: {viento} km/h")
print(f"☁️ Condiciones: {descripcion}")
print(f"📍 Ubicación: Madrid (40.42°N, 3.70°W)")



### ✅ Conceptos clave sobre APIs:

1️⃣ **APIs** = Servicios web que devuelven datos cuando les haces una petición

2️⃣ **URLs de API** tienen 3 partes:
   - 🌐 Servidor: `https://api.open-meteo.com`
   - 📂 Ruta: `/v1/forecast`
   - ❓ Parámetros: `?latitude=40.4168&longitude=-3.7038`

3️⃣ **HTTP GET** = Método para solicitar información (como hacer una pregunta)

4️⃣ **JSON** = Formato de texto estructurado para intercambiar datos
   - Se convierte a diccionario Python con `.json()`
   - Una vez recibes la respuesta de la API se puede formatear para consumo final (que quede bonita) o para que lo consuma otro servicio de la aplicación (otro JSON por ejemplo)

5️⃣ **Códigos de estado HTTP**:
   - `200` = Éxito
   - `404` = No encontrado  
   - `500` = Error del servidor


### 🤖 Conexión con la IA

**Las APIs de IA funcionan EXACTAMENTE IGUAL**, pero en lugar de devolver el clima, generan texto nuevo basándose en tu prompt.

**Flujo típico**:
```
Tú → Envías pregunta (prompt) → API de IA → Recibe → Procesa → Devuelve respuesta (JSON) → Tú
```

🔜 **A continuación**: Tu primera llamada a una IA generativa real con Gemini!


---
# 2. PRIMERAS LLAMADAS A APIS DE IA

Ahora vamos a hacer llamadas REALES a modelos de IA generativa. Veremos diferentes formas de hacerlo.

⚠️ **Recuerda**: Necesitas API keys (configuradas en la sección 0). Si no las tienes, puedes leer el código para aprender.
**Obtener API key GRATIS**: https://makersuite.google.com/app/apikey

**Empezamos con Gemini porque es GRATIS** (número de peticiones por minuto limitadas).

El **modelo** es la "versión" o "tamaño" del cerebro de la IA. Vamos a usar:

- `gemini-3-flash-preview`: Rápido, económico y ultra-actualizado (GRATIS) - Cada modelo tiene diferentes capacidades y costes

## 2.1 Ejercicio: Tu primera llamada a IA Generativa

Vamos a hacer una petición HTTP a Gemini, igual que hicimos con la API del tiempo, pero ahora para **generar texto**.

**¿Qué vamos a hacer?**
1. 📤 Enviar una pregunta simple a Gemini
2. 📥 Recibir la respuesta en formato JSON (raw)
3. 🔍 Analizar la estructura del JSON

**Pregunta al modelo**: "Explica qué es la IA generativa en una frase"

### 📡 PASO 0: Definir el modelo con el que vamos a trabajar

In [ ]:
#modelo = "gemini-3-flash-preview"
modelo = "gemini-2.5-flash-lite"

### 📡 PASO 1: Preparar la petición a Gemini

In [ ]:

# Recuperar la API Key del entorno
api_key = os.getenv('GEMINI_API_KEY')

if not api_key:
    print("⚠️ No se encontró API Key de Gemini")
    print("💡 Puedes continuar leyendo el código para aprender")
else:
    print("✅ API Key encontrada")

# Construir la URL de la API REST de Gemini

url = f"https://generativelanguage.googleapis.com/v1beta/models/{modelo}:generateContent?key={api_key}"

print(f"\n🌐 Servidor: generativelanguage.googleapis.com")
print(f"📦 Modelo: {modelo}")
print(f"🔧 Método HTTP: POST (para enviar datos)")

# --------------------------------------------------
# PREPARAR EL PAYLOAD (datos que enviamos para hacer la petición o preguntar algo a Gemini)
# --------------------------------------------------

# El "payload" es el cuerpo de la petición - lo que le enviamos a la API
# Ahora incluimos DOS partes:
#   1. systemInstruction = Instrucciones globales (🎭 ROL del modelo)
#   2. contents = Conversación (💬 PREGUNTA del usuario)

payload = {
    # 🎭 SYSTEM INSTRUCTION: Define cómo debe comportarse el modelo
    "systemInstruction": {
        "parts": [
            {
                "text": "Eres un profesor experto en tecnología que explica conceptos complejos de forma clara y concisa. Usa analogías simples cuando sea posible."
            }
        ]
    },
    
    # 💬 CONTENTS: La conversación (mensajes del usuario)
    "contents": [
        {
            "parts": [
                {
                    "text": "Explica qué es la IA generativa en una frase"
                }
            ]
        }
    ]
}

print("\n📋 Payload (datos que enviamos):")
print(json.dumps(payload, indent=2, ensure_ascii=False))

print("\n💡 IMPORTANTE - Estructura del payload:")
print("   🎭 systemInstruction = Define ROL y COMPORTAMIENTO del modelo")
print("   💬 contents = La PREGUNTA o conversación del usuario")
print("   ↪️ El system prompt afecta CÓMO responde, no QUÉ responde")

# Headers: información adicional que necesita la API
headers = {
    "Content-Type": "application/json"  # Le decimos que enviamos JSON
}

print("\n🎯 Headers:")
print(json.dumps(headers, indent=2))

print("\n⏳ Listo para enviar la petición...")

### 📡 PASO 2: Enviar la petición y ver el JSON completo (raw)

Ahora vamos a enviar la petición con `requests.post()` y veremos la respuesta en **DOS formatos**:

1. **📄 RAW** (texto plano): Lo que realmente viaja por internet
2. **📦 JSON formateado**: Estructura visual para explorar los datos

**💡 ¿Por qué ver ambos?**

| Formato | Qué enseña | Por qué es importante |
|---------|------------|----------------------|
| **RAW** | Texto plano sin procesar | Entender que las APIs devuelven strings, no objetos mágicos |
| **JSON** | Estructura jerárquica visual | Identificar cómo acceder a los datos que necesitas |

**🎓 Concepto clave**: `response.text` (raw) → `response.json()` → diccionario Python

In [ ]:
# Intentar hacer la petición
try:
    print("📡 Enviando petición a Gemini...")
    
    # POST = Enviar datos al servidor (no solo leer, como GET)
    # verify=VERIFICAR_SSL = controla verificación de certificados SSL
    # timeout=60 = esperar hasta 60 segundos (para conexiones lentas/proxy)
    response = requests.post(url, headers=headers, json=payload, timeout=90, verify=VERIFICAR_SSL)
    
    print(f"\n✅ Respuesta recibida!")
    print(f"📊 Código HTTP: {response.status_code}")
    
    if response.status_code == 200:
        # --------------------------------------------------
        # PASO 2A: VER LA RESPUESTA RAW (texto plano)
        # --------------------------------------------------
        
        print("\n" + "="*60)
        print("📄 RESPUESTA RAW (texto plano sin procesar):")
        print("="*60)
        print("\n💡 Esto es lo que REALMENTE viaja por internet:")
        print("   - Texto plano (string)")
        print("   - Formato JSON (aún no parseado)")
        print("   - Así llega desde el servidor de Google\n")
        
        # Mostrar los primeros 800 caracteres del texto raw
        texto_raw = response.text
        print(texto_raw[:800])
        
        if len(texto_raw) > 800:
            print(f"\n... (truncado, total: {len(texto_raw)} caracteres)")
        
        print("\n" + "="*60)
        print("🔄 PARSEANDO JSON (convirtiendo texto a diccionario Python)...")
        print("="*60)
        
        # --------------------------------------------------
        # PASO 2B: CONVERTIR A JSON (diccionario Python)
        # --------------------------------------------------
        
        # .json() convierte el texto raw en un diccionario Python
        json_response = response.json()
        
        print("✅ Conversión exitosa: texto → diccionario Python")
        print("💡 Ahora podemos acceder a los datos con: json_response['clave']")
        
        print("\n" + "="*60)
        print("📦 JSON FORMATEADO (estructura visual e interactiva):")
        print("="*60)
        print("\n💡 Ventajas del JSON formateado:")
        print("   - Ver la jerarquía de datos (objetos, arrays)")
        print("   - Explorar y colapsar secciones")
        print("   - Identificar qué datos necesitamos extraer\n")
        
        # Mostrar el JSON completo de forma interactiva
        from IPython.display import JSON as DisplayJSON
        display(DisplayJSON(json_response, expanded=True))
        
        print("\n" + "="*60)
        print("🔍 ANALIZANDO LA ESTRUCTURA DEL JSON:")
        print("="*60)
        
        # Explicar cada parte del JSON
        print("\n1️⃣ Claves principales del JSON:")
        for key in json_response.keys():
            print(f"   - '{key}'")
        
        print("\n2️⃣ Candidatos (posibles respuestas que generó el modelo):")
        num_candidatos = len(json_response.get('candidates', []))
        print(f"   - Total de candidatos: {num_candidatos}")
        print(f"   💡 Por defecto la API genera 1 candidato (la mejor respuesta)")
        print(f"   💡 Puedes pedir más con 'candidateCount' en el payload")
        
        if 'candidates' in json_response and len(json_response['candidates']) > 0:
            candidato = json_response['candidates'][0]
            print(f"\n3️⃣ Estructura del primer candidato (candidates[0]):")
            print(f"   - Claves: {list(candidato.keys())}")
            print(f"   💡 Siempre usamos candidates[0] porque es la mejor respuesta")
            
            if 'content' in candidato:
                print(f"\n4️⃣ Contenido del candidato (candidates[0]['content']):")
                print(f"   - Claves: {list(candidato['content'].keys())}")
                print(f"   - Rol: {candidato['content'].get('role', 'N/A')}")
                print(f"   - Número de partes: {len(candidato['content'].get('parts', []))}")
                print(f"   💡 'parts' contiene los fragmentos de texto generados")
        
        print("\n🎯 Ruta para extraer el texto final:")
        print("   json_response['candidates'][0]['content']['parts'][0]['text']")
        print("\n✅ Resumen de lo aprendido:")
        print("   📄 Raw → Vimos el texto plano que viaja por internet")
        print("   📦 JSON → Exploramos la estructura de datos")
        print("   🔍 Análisis → Identificamos dónde está la respuesta")
        print("\n💡 En la siguiente celda extraeremos el texto de forma limpia...")
        
    else:
        print(f"\n❌ Error HTTP {response.status_code}")
        print(f"Respuesta: {response.text[:200]}")
        
except Exception as e:
    print(f"\n❌ Error: {type(e).__name__}")
    print(f"Detalles: {e}")
    print("\n💡 Verifica que tu API Key sea correcta y tengas conexión a internet")

### 🎯 PASO 3: Parsear el JSON para obtener la respuesta de forma amigable



Ahora que vimos el JSON completo, vamos a **extraer solo el texto de la respuesta** y mostrarlo de forma legible.

#### 🤔 ¿Qué son los "candidatos" (candidates)?

Los **candidatos** son las posibles respuestas que la IA genera. ¿Por qué múltiples?

**Explicación simple**: La IA genera texto de forma probabilística (basada en probabilidades). Cada vez que genera una respuesta, puede haber varias opciones válidas con diferentes "caminos" de generación.

| Concepto | Explicación |
|----------|-------------|
| **candidates** | Array con las respuestas generadas por el modelo |
| **candidates[0]** | Primera respuesta (la mejor según el modelo) |
| **candidates[1]** | Segunda alternativa (si la solicitas) |
| **¿Cuántos?** | Por defecto: 1. Puedes pedir más con `candidateCount: 3` en el payload |

**Ejemplo práctico**:
```python
# Petición normal → 1 candidato
payload = {"contents": [...]}

# Pedir 3 alternativas → 3 candidatos
payload = {
    "contents": [...],
    "generationConfig": {
        "candidateCount": 3  # Genera 3 respuestas diferentes
    }
}
```

**¿Cuál usar?** 
- 🎯 **Normalmente**: Siempre `candidates[0]` (la mejor respuesta)
- 🎲 **Múltiples opciones**: Úsalos todos si quieres dar opciones al usuario

---

**Ruta completa en el JSON**: 
```
candidates[0] → content → parts[0] → text
    ↓           ↓          ↓           ↓
 Primera    Contenido  Primera    Texto
 respuesta  generado   parte      final
```

In [ ]:
# Si la petición anterior fue exitosa, parseamos la respuesta
try:
    if response.status_code == 200:
        json_response = response.json()
        
        print("🔍 EXTRAYENDO LA RESPUESTA DEL JSON...")
        print("="*60)
        
        # Navegar por la estructura del JSON
        print("\n📍 Ruta: json_response['candidates'][0]['content']['parts'][0]['text']")
        
        # Extraer el texto paso a paso (para que veas cada nivel)
        candidatos = json_response['candidates']
        print(f"\n1️⃣ Candidatos encontrados: {len(candidatos)}")
        
        primer_candidato = candidatos[0]
        print(f"2️⃣ Usando el primer candidato")
        
        contenido = primer_candidato['content']
        print(f"3️⃣ Contenido extraído (rol: {contenido.get('role')})")
        
        partes = contenido['parts']
        print(f"4️⃣ Partes del contenido: {len(partes)}")
        
        texto_respuesta = partes[0]['text']
        print(f"5️⃣ Texto extraído ✅")
        
        # Mostrar la respuesta de forma bonita
        print("\n" + "="*60)
        print("🤖 RESPUESTA DE GEMINI (LIMPIA Y LEGIBLE):")
        print("="*60)
        print()
        print(texto_respuesta)
        print()
        print("="*60)
        
        # Información adicional útil
        if 'usageMetadata' in json_response:
            metadata = json_response['usageMetadata']
            print(f"\n📊 Metadatos de uso:")
            print(f"   - Tokens del prompt: {metadata.get('promptTokenCount', 'N/A')}")
            print(f"   - Tokens de la respuesta: {metadata.get('candidatesTokenCount', 'N/A')}")
            print(f"   - Tokens totales (incluyendo razonamiento): {metadata.get('totalTokenCount', 'N/A')}")
        
        print("\n✨ ¡Listo! Has completado tu primera llamada a IA Generativa")
        
    else:
        print("⚠️ No hay respuesta exitosa para parsear")
        
except NameError:
    print("⚠️ Primero ejecuta la celda anterior para hacer la petición")
except KeyError as e:
    print(f"❌ Error al acceder a la clave: {e}")
    print("💡 La estructura del JSON puede haber cambiado")
except Exception as e:
    print(f"❌ Error: {type(e).__name__}")
    print(f"Detalles: {e}")

### 🎲 PASO 4: Experimentar con Temperature (Creatividad)

#### 🌡️ ¿Qué es la Temperature?

Ahora vamos a explorar uno de los parámetros más importantes: **temperature** (temperatura).


La **temperatura** controla la **aleatoriedad y creatividad** de las respuestas del modelo.

**Analogía simple**: Imagina que le preguntas a 3 personas diferentes "¿Qué es la IA?" con diferentes personalidades:
- 🎯 **Temperature baja (0.1)**: Persona muy seria y técnica → Siempre responde igual
- ⚖️ **Temperature media (1.0)**: Persona equilibrada → Varía un poco las respuestas
- 🎨 **Temperature alta (1.7)**: Persona creativa e inesperada → Cada respuesta es muy diferente

##### 📊 Cómo funciona técnicamente:

Cuando la IA genera texto, calcula **probabilidades** para cada palabra posible:

```
Prompt: "El cielo es..."
Probabilidades originales:
- "azul" → 60%
- "hermoso" → 20%
- "infinito" → 10%
- "grande" → 5%
- "mágico" → 5%
```

**Temperature modifica estas probabilidades**:

| Temperature | Efecto | Resultado |
|-------------|--------|----------|
| **0.0 - 0.3** | 🎯 Elige siempre la más probable | "azul" en el 99% de los casos |
| **0.7 - 1.0** | ⚖️ Balance probabilidades | "azul" 70%, otras 30% |
| **1.5 - 2.0** | 🎨 Distribuye uniformemente | Cualquier opción puede salir |

##### 🎯 Tabla de uso recomendado:

| Temperature | Comportamiento | Casos de uso | Ejemplo |
|-------------|----------------|--------------|----------|
| **0.0 - 0.3** | Determinista, predecible | Código, matemáticas, traducciones | "¿Cuál es 2+2?" → Siempre "4" |
| **0.4 - 0.7** | Ligeramente creativo | Documentación, emails profesionales | Variaciones sutiles |
| **0.8 - 1.2** | Equilibrado | Marketing, contenido general | Creatividad controlada |
| **1.3 - 1.7** | Muy creativo | Brainstorming, títulos, slogans | Respuestas inesperadas |
| **1.8 - 2.0** | Caótico, experimental | Generación artística, poesía | Máxima aleatoriedad |


**💡 Vamos a probarlo**: Haremos la **misma pregunta 3 veces** con diferentes temperaturas:
- 🎯 **Temperature 0.1**: Respuesta muy consistente y predecible
- ⚖️ **Temperature 1.0**: Respuesta equilibrada y natural
- 🎨 **Temperature 1.7**: Respuesta muy creativa e inesperada

In [ ]:
# --------------------------------------------------
# EXPERIMENTO: Comparar 3 temperaturas diferentes
# --------------------------------------------------

# Verificar API key
if not api_key:
    print("⚠️ Necesitas configurar tu API Key de Gemini primero")
else:
    print("🌡️ EXPERIMENTO: Misma pregunta, 3 temperaturas diferentes")
    print("="*60)
    print("\n📝 Pregunta: 'Escribe un eslogan creativo para una empresa de IA en 5 palabras'\n")
    
    # Definir las 3 temperaturas a probar
    temperaturas = [0.1, 1.0, 1.7]
    
    # Diccionario para almacenar las respuestas
    respuestas_temperatura = {}
    
    # Hacer 3 peticiones con diferentes temperaturas
    for temp in temperaturas:
        print(f"\n{'='*60}")
        print(f"🌡️ Probando con TEMPERATURE = {temp}")
        print(f"{'='*60}")
        
        # Preparar el payload con la temperatura específica
        payload_temp = {
            "systemInstruction": {
                "parts": [{"text": "Eres un experto en marketing y creatividad que genera slogans impactantes."}]
            },
            "contents": [{
                "parts": [{"text": "Escribe un eslogan creativo para una empresa de IA en 5 palabras. Escribe solo las 5 palabras."}]
            }],
            "generationConfig": {
                "temperature": temp,
                "candidateCount": 1
            }
        }
        
        try:
            print(f"⏳ Enviando petición con temperature={temp}...")
            resp = requests.post(url, headers=headers, json=payload_temp, timeout=60, verify=VERIFICAR_SSL)
            
            if resp.status_code == 200:
                json_resp = resp.json()
                texto = json_resp['candidates'][0]['content']['parts'][0]['text']
                respuestas_temperatura[temp] = texto
                
                print(f"\n✅ Respuesta recibida:")
                print(f"   {texto}")
                
                # Mostrar tokens si disponible
                if 'usageMetadata' in json_resp:
                    tokens = json_resp['usageMetadata'].get('totalTokenCount', 'N/A')
                    print(f"\n📊 Tokens consumidos: {tokens}")
            else:
                print(f"❌ Error HTTP {resp.status_code}")
                respuestas_temperatura[temp] = f"[Error: {resp.status_code}]"
        except Exception as e:
            print(f"❌ Error: {type(e).__name__}")
            print(f"   {e}")
            respuestas_temperatura[temp] = "[Error de conexión]"
        
        # Pequeña pausa entre peticiones
        if temp != temperaturas[-1]:
            time.sleep(1)
    
    print("\n" + "="*60)
    print("✅ Experimento completado")
    print("💡 En la siguiente celda compararemos los resultados")
    print("="*60)

### 📊 PASO 5: Comparar las respuestas con diferente temperatura

Ahora vamos a visualizar las **3 respuestas lado a lado** para ver claramente el efecto de la temperatura.

In [ ]:
# --------------------------------------------------
# COMPARACIÓN VISUAL DE LAS 3 TEMPERATURAS
# --------------------------------------------------

if 'respuestas_temperatura' in locals() and len(respuestas_temperatura) == 3:
    print("\n" + "="*60)
    print("🔍 COMPARACIÓN: Mismo prompt, 3 temperaturas diferentes")
    print("="*60)
    print("\n📝 Pregunta: 'Escribe un eslogan creativo para una empresa de IA en 5 palabras'\n")
    
    # Mostrar cada respuesta en un cuadro visual
    for temp, respuesta in sorted(respuestas_temperatura.items()):
        # Determinar el emoji y descripción según la temperatura
        if temp <= 0.3:
            emoji = "🎯"
            desc = "DETERMINISTA"
            caracteristica = "Muy predecible, consistente"
        elif temp <= 1.2:
            emoji = "⚖️"
            desc = "EQUILIBRADA"
            caracteristica = "Balance creatividad/coherencia"
        else:
            emoji = "🎨"
            desc = "CREATIVA"
            caracteristica = "Muy variada, inesperada"
        
        print("┌" + "─"*58 + "┐")
        print(f"│ {emoji} TEMPERATURE = {temp:<6} ({desc}){' '*(36-len(desc))}│")
        print(f"│ {caracteristica}{' '*(56-len(caracteristica))}│")
        print("├" + "─"*58 + "┤")
        
        # Dividir el texto en líneas que quepan (max 54 caracteres por línea)
        palabras = respuesta.strip().split()
        linea_actual = "│ "
        for palabra in palabras:
            if len(linea_actual) + len(palabra) + 1 <= 57:
                linea_actual += palabra + " "
            else:
                # Completar la línea con espacios
                linea_actual += " " * (58 - len(linea_actual)) + "│"
                print(linea_actual)
                linea_actual = "│ " + palabra + " "
        
        # Última línea
        if linea_actual != "│ ":
            linea_actual += " " * (58 - len(linea_actual)) + "│"
            print(linea_actual)
        
        print("└" + "─"*58 + "┘")
        print()
    
    # Análisis comparativo
    print("="*60)
    print("📊 ANÁLISIS COMPARATIVO:")
    print("="*60)
    
    for temp, respuesta in sorted(respuestas_temperatura.items()):
        num_palabras = len(respuesta.split())
        num_caracteres = len(respuesta)
        
        if temp <= 0.3:
            observacion = "Respuesta más técnica y directa"
        elif temp <= 1.2:
            observacion = "Respuesta natural y equilibrada"
        else:
            observacion = "Respuesta más creativa y original"
        
        print(f"\n🌡️ Temperature {temp}:")
        print(f"   📏 Palabras: {num_palabras}")
        print(f"   🔤 Caracteres: {num_caracteres}")
        print(f"   💡 Observación: {observacion}")
    
    print("\n" + "="*60)
    print("🎓 CONCLUSIONES:")
    print("="*60)
    print("\n✅ Temperature BAJA (0.1):")
    print("   - Respuestas predecibles y consistentes")
    print("   - Ideal para: Código, datos técnicos, traducciones")
    print("   - Si ejecutas varias veces, la respuesta será casi idéntica")
    
    print("\n✅ Temperature MEDIA (1.0):")
    print("   - Balance entre creatividad y coherencia")
    print("   - Ideal para: Marketing, contenido general, emails")
    print("   - Cada ejecución da respuestas diferentes pero relacionadas")
    
    print("\n✅ Temperature ALTA (1.7):")
    print("   - Máxima creatividad e imprevisibilidad")
    print("   - Ideal para: Brainstorming, slogans, contenido artístico")
    print("   - Cada ejecución puede dar respuestas muy diferentes")
    
    print("\n💡 RECOMENDACIÓN PRÁCTICA:")
    print("   Ejecuta la celda del PASO 4 varias veces para ver")
    print("   cómo varía la respuesta con cada temperatura")
    
else:
    print("⚠️ Primero ejecuta la celda del PASO 4 para generar las respuestas")


## 2.2 Ejercicio: Simulando una conversacion con memoria

### Introducción

Hasta ahora, cada llamada a la API ha sido **independiente** - el modelo NO recuerda nada de peticiones anteriores.

#### ❌ El Problema: Sin Memoria

```
Tú: "¿Cuál es la capital de Francia?"
IA: "París"

Tú: "¿Cuántos habitantes tiene?"
IA: "¿De qué ciudad hablas?" ← NO RECUERDA "París"
```

#### ✅ La Solución: Enviar el Historial Completo

**Concepto clave**: Cada llamada a la API debe incluir **TODA la conversación anterior** en el payload.

```python
# Primera llamada
contenido = [
    {"role": "user", "parts": [{"text": "¿Cuál es la capital de Francia?"}]}
]

# Segunda llamada - incluye TODA la conversación anterior
contenido = [
    {"role": "user", "parts": [{"text": "¿Cuál es la capital de Francia?"}]},
    {"role": "model", "parts": [{"text": "París"}]},
    {"role": "user", "parts": [{"text": "¿Cuántos habitantes tiene?"}]}
]
```

---

#### 📊 Estructura de una Conversación con Contexto

| Turno | Role | Contenido | Estado del historial |
|-------|------|-----------|---------------------|
| 1️⃣ | user | "¿Capital de Francia?" | [user1] |
| 2️⃣ | model | "París" | [user1, model1] |
| 3️⃣ | user | "¿Cuántos habitantes?" | [user1, model1, user2] |
| 4️⃣ | model | "2.2 millones" | [user1, model1, user2, model2] |

**💡 Importante**: 
- Los modelos son **stateless** (sin estado) - no guardan nada entre llamadas
- **Tú** eres responsable de mantener el historial en tu código
- Cada llamada cuesta más tokens cuanto más largo sea el historial

---

#### 🔄 Diagrama de Flujo: Conversación con Memoria

```
┌─────────────────────────────────────────────────────────┐
│  LLAMADA 1: "¿Capital de Francia?"                      │
├─────────────────────────────────────────────────────────┤
│  Historial: []                                          │
│  Enviar: [user: "¿Capital de Francia?"]                 │
│  Respuesta: "París"                                     │
│  Guardar en historial ✅                                │
└─────────────────────────────────────────────────────────┘
                       ↓
┌─────────────────────────────────────────────────────────┐
│  LLAMADA 2: "¿Cuántos habitantes tiene?"                │
├─────────────────────────────────────────────────────────┤
│  Historial: [user: "¿Capital?", model: "París"]        │
│  Enviar: [user: "¿Capital?", model: "París",           │
│           user: "¿Cuántos habitantes?"]                 │
│  Respuesta: "París tiene 2.2 millones"                 │
│  Guardar en historial ✅                                │
└─────────────────────────────────────────────────────────┘
                       ↓
┌─────────────────────────────────────────────────────────┐
│  LLAMADA 3: "¿Y el país?"                               │
├─────────────────────────────────────────────────────────┤
│  Historial: [user+model de llamada 1 y 2]              │
│  Enviar: TODO el historial + nuevo mensaje             │
│  Respuesta: "París es la capital de Francia"           │
└─────────────────────────────────────────────────────────┘
```

**🎯 En resumen**: Para simular "memoria", enviamos toda la conversación en cada llamada.

**Objetivo**: Crear una conversación coherente donde la IA recuerda lo que dijiste antes.

**Escenario**: Vamos a planificar unas vacaciones:
1. Preguntamos sobre un destino
2. Hacemos preguntas de seguimiento que requieren contexto
3. La IA responde coherentemente gracias al historial

**🎯 Lo que aprenderás**:
- Cómo mantener un historial de conversación
- Cómo estructurar mensajes con roles (user/model)
- Cómo enviar el contexto completo en cada llamada

### 🔧 PASO 1: Configurar el sistema y definir función auxiliar

In [ ]:
# --------------------------------------------------
# CONFIGURACIÓN INICIAL
# --------------------------------------------------

# Verificar API key
api_key_conv = os.getenv('GEMINI_API_KEY')

if not api_key_conv:
    print("⚠️ No se encontró API Key de Gemini")
    print("💡 Configura tu API key en la sección 0 para ejecutar este ejercicio")
else:
    print("✅ API Key encontrada")
    
    # URL de la API
    url_conv = f"https://generativelanguage.googleapis.com/v1beta/models/{modelo}:generateContent?key={api_key_conv}"
    
    # --------------------------------------------------
    # INICIALIZAR HISTORIAL DE CONVERSACIÓN
    # --------------------------------------------------
    
    # 💡 CLAVE: Este array almacenará TODA la conversación
    historial_conversacion = []
    
    print(f"📦 Modelo: {modelo}")
    print(f"🗂️ Historial inicializado (vacío por ahora)")
    print(f"📊 Mensajes en historial: {len(historial_conversacion)}")
    
    # --------------------------------------------------
    # FUNCIÓN AUXILIAR: Enviar mensaje y actualizar historial
    # --------------------------------------------------
    
    def enviar_mensaje(mensaje_usuario, historial):
        """
        Envía un mensaje a Gemini incluyendo todo el historial.
        
        Args:
            mensaje_usuario (str): El nuevo mensaje del usuario
            historial (list): Lista con toda la conversación previa
            
        Returns:
            tuple: (respuesta_texto, historial_actualizado)
        """
        
        # 1. Añadir el nuevo mensaje del usuario al historial
        historial.append({
            "role": "user",
            "parts": [{"text": mensaje_usuario}]
        })
        
        print(f"\n{'='*60}")
        print(f"📤 ENVIANDO MENSAJE (con {len(historial)} mensajes en historial)")
        print(f"{'='*60}")
        print(f"💬 Usuario: {mensaje_usuario}")
        
        # 2. Preparar el payload CON TODO EL HISTORIAL
        payload = {
            "systemInstruction": {
                "parts": [{"text": "Eres un asistente de viajes amigable y conocedor que ayuda a planificar vacaciones."}]
            },
            "contents": historial  # 🎯 Aquí va TODO el historial
        }
        
        print(f"\n📊 Estado del historial:")
        print(f"   - Total de mensajes enviados: {len(historial)}")
        print(f"   - Último mensaje: {mensaje_usuario[:50]}...")
        
        try:
            # 3. Hacer la petición HTTP (con verify=VERIFICAR_SSL para soporte de proxy)
            response = requests.post(url_conv, headers={"Content-Type": "application/json"}, json=payload, timeout=30, verify=VERIFICAR_SSL)
            
            if response.status_code == 200:
                json_resp = response.json()
                
                # 4. Extraer la respuesta
                respuesta_texto = json_resp['candidates'][0]['content']['parts'][0]['text']
                
                # 5. Añadir la respuesta del modelo al historial
                historial.append({
                    "role": "model",
                    "parts": [{"text": respuesta_texto}]
                })
                
                print(f"\n✅ Respuesta recibida")
                print(f"🤖 IA: {respuesta_texto}")
                print(f"\n📊 Historial actualizado: {len(historial)} mensajes totales")
                
                # Mostrar consumo de tokens si está disponible
                if 'usageMetadata' in json_resp:
                    tokens = json_resp['usageMetadata'].get('totalTokenCount', 'N/A')
                    print(f"🔢 Tokens consumidos: {tokens}")
                    print(f"   💡 Cuanto más largo el historial, más tokens se consumen")
                
                return respuesta_texto, historial
                
            else:
                print(f"\n❌ Error HTTP {response.status_code}")
                print(f"Respuesta: {response.text[:300]}")
                return None, historial
                
        except Exception as e:
            print(f"\n❌ Error: {type(e).__name__}")
            print(f"Detalles: {e}")
            return None, historial
    
    print("\n" + "="*60)
    print("✅ Sistema configurado y función auxiliar creada")
    print("🚀 ¡Listo para empezar la conversación!")
    print("="*60)

### 💬 PASO 2: Primera pregunta (Turno 1)

In [ ]:
# Turno 1: Pregunta inicial sobre un destino
if api_key_conv:
    _, historial_conversacion = enviar_mensaje(
        "Recomiéndame un destino de playa en España para ir en verano",
        historial_conversacion
    )

### 💬 PASO 3: Segunda pregunta (Turno 2) - Requiere contexto

In [ ]:
# Turno 2: Pregunta de seguimiento que necesita contexto
if api_key_conv:
    _, historial_conversacion = enviar_mensaje(
        "¿Qué temperatura hace ahí en agosto?",  # "ahí" = referencia al destino anterior
        historial_conversacion
    )
    
    print("\n💡 OBSERVA: La IA entiende 'ahí' porque enviamos el historial completo")

### 💬 PASO 4: Tercera pregunta (Turno 3) - Más contexto

In [ ]:
# Turno 3: Otra pregunta de seguimiento
if api_key_conv:
    _, historial_conversacion = enviar_mensaje(
        "¿Qué actividades acuáticas puedo hacer allí?",  # "allí" = referencia contextual
        historial_conversacion
    )

### 📊 PASO 5: Visualizar el historial completo

In [ ]:
# Mostrar el historial completo de forma estructurada
if api_key_conv:
    print("\n" + "="*60)
    print("📚 HISTORIAL COMPLETO DE LA CONVERSACIÓN")
    print("="*60)
    print(f"\n📊 Total de mensajes intercambiados: {len(historial_conversacion)}")
    print(f"💬 Turnos de conversación: {len(historial_conversacion) // 2}")
    
    print("\n🔍 Desglose mensaje por mensaje:\n")
    
    for i, mensaje in enumerate(historial_conversacion, 1):
        rol = mensaje['role']
        texto = mensaje['parts'][0]['text']
        
        # Formato visual diferente para user vs model
        if rol == "user":
            icono = "👤"
            etiqueta = "USUARIO"
        else:
            icono = "🤖"
            etiqueta = "IA"
        
        print(f"{icono} {etiqueta} (Mensaje #{i}):")
        print(f"   {texto[:150]}{'...' if len(texto) > 150 else ''}")
        print()
    
    print("="*60)
    print("✅ CONCEPTO CLAVE DEMOSTRADO:")
    print("="*60)
    print("✅ Cada mensaje incluye TODO el historial anterior")
    print("✅ La IA no 'recuerda' - nosotros le enviamos el contexto")
    print("✅ Cuanto más larga la conversación, más tokens se consumen")
    print("✅ Los roles 'user' y 'model' estructuran la conversación")
    print("\n💡 Este es el patrón básico de TODOS los chatbots con IA")

---
## 💡 Consejo Avanzado: Experimenta con Diferentes Modelos  

¿Quieres ver cómo **diferentes modelos** generan respuestas distintas a las mismas preguntas?

### 🔄 Prueba esto:

1. **Ve al PASO 0 del Ejercicio 2.1** (cerca del principio del notebook)
2. **Cambia el modelo** de:
   ```python
   modelo = "gemini-2.5-flash-lite"
   ```
   a otro modelo disponible:
   ```python
   modelo = "gemini-3-flash-preview"    # Más rápido y actualizado
   # modelo = "gemini-2.5-flash"        # Versión intermedia
   # modelo = "gemini-2.5-pro"           # Mayor capacidad de razonamiento
   ```

3. **Ejecuta TODO el notebook de nuevo** (Restart & Run All)
4. **Observa las diferencias** en:
   - 📝 Estilo de escritura
   - 🎯 Precisión de las respuestas
   - 🎨 Creatividad (especialmente en el experimento de temperatura)
   - ⚡ Velocidad de respuesta
   - 🪙 Consumo de tokens

### ✨ ¿Qué aprenderás?

- Cada modelo tiene **personalidad y capacidades diferentes**
- Los modelos más grandes (`pro`) son más precisos pero consumen más tokens
- Los modelos `flash/lite` son ideales para prototipos rápidos
- La elección del modelo depende de tu caso de uso (velocidad vs calidad)

**💡 Tip**: Guarda una copia de los resultados de cada modelo para comparar lado a lado.


---
# 3. 📝 RESUMEN FINAL: Lo que has aprendido en este notebook

¡Felicidades! 🎉 Has completado la introducción práctica a IA Generativa. Este es un resumen de todo lo que has aprendido.

## 🎓 Conceptos Fundamentales

### 1. ¿Qué es IA Generativa?

✅ **Aprendiste**: La IA Generativa **crea contenido nuevo** (texto, imágenes, audio) en lugar de solo analizar datos existentes.

✅ **Diferencia clave**: 
- **IA Tradicional**: Clasifica/analiza datos existentes
- **IA Generativa**: Crea contenido nuevo basándose en patrones aprendidos

### 2. Modelos de Lenguaje (LLMs)

✅ **Aprendiste**: Los modelos como GPT-5.2, Gemini 3, Claude 4.5, Qwen3, DeepSeek R2 son redes neuronales entrenadas con billones de palabras.

✅ **Conceptos clave**:
- **Parámetros**: Tamaño del modelo (más parámetros = más capacidad)
- **Contexto/Context Window**: Cuánto texto puede "recordar" (hasta 4M tokens en Gemini 3 Pro)
- **Tokens**: Unidades básicas de texto (~1 token ≈ 0.75 palabras)
- **MoE (Mixture of Experts)**: Arquitectura eficiente que activa solo parte de los parámetros

### 3. APIs y Peticiones HTTP

✅ **Aprendiste**: Las APIs son interfaces que permiten comunicarte con servicios de IA mediante peticiones HTTP.

✅ **Flujo completo**:
```
Tu código → HTTP Request → Servidor IA → Modelo LLM → Respuesta → Tu código
```

✅ **Métodos HTTP**:
- **GET**: Solicitar información (ej: API del tiempo)
- **POST**: Enviar datos para procesar (ej: generar texto con IA)

✅ **Componentes de una petición**:
- **URL**: Dirección del servicio (`https://api.servicio.com/endpoint`)
- **Headers**: Metadatos (`Content-Type: application/json`)
- **Payload/Body**: Datos enviados (tu pregunta, configuración)
- **Response**: Lo que devuelve el servidor (JSON con la respuesta)

---

## 🔧 Habilidades Técnicas Adquiridas

### 4. Trabajar con JSON

✅ **Aprendiste**: JSON es el formato estándar para intercambiar datos con APIs.

✅ **Conceptos prácticos**:
- **Texto RAW** → Lo que realmente viaja por internet (string)
- **Parsing**: Convertir texto JSON a diccionario Python con `.json()`
- **Navegar estructuras**: `json['candidates'][0]['content']['parts'][0]['text']`

### 5. System Prompts vs User Prompts

✅ **Aprendiste**: Los mensajes tienen diferentes roles con diferentes propósitos.

| Tipo | Propósito | Ejemplo |
|------|-----------|---------|
| **system** | Instrucciones globales, personalidad | "Eres un profesor que explica de forma simple" |
| **user** | Preguntas del usuario | "¿Qué es la IA?" |
| **model/assistant** | Respuestas de la IA | "La IA es..." |

✅ **Por qué importa**: El system prompt define **CÓMO** responde la IA, no **QUÉ** responde.

### 6. Parámetros de Generación

✅ **Aprendiste**: Puedes controlar cómo genera texto la IA con parámetros.

#### 🌡️ **Temperature** (0.0 - 2.0)

| Valor | Comportamiento | Cuándo usar |
|-------|----------------|-------------|
| **0.0** | Determinista, siempre igual | Código, matemáticas, hechos |
| **0.7** | Equilibrado (por defecto) | Chat general, emails |
| **1.0-1.5** | Creativo, variado | Brainstorming, contenido artístico |


### 7. Contexto y Memoria en Conversaciones

✅ **Aprendiste**: Los modelos de IA son **stateless** (sin memoria entre llamadas).

✅ **Solución**: Enviar **todo el historial** de la conversación en cada petición.

```python
# Estructura de una conversación
historial = [
    {"role": "user", "parts": [{"text": "Pregunta 1"}]},
    {"role": "model", "parts": [{"text": "Respuesta 1"}]},
    {"role": "user", "parts": [{"text": "Pregunta 2"}]},
    {"role": "model", "parts": [{"text": "Respuesta 2"}]},
    # ... y así sucesivamente
]
```

✅ **Implicaciones**:
- 💰 Más historial = más tokens = más coste
- 🧠 Tú eres responsable de mantener el historial
- 📏 El historial está limitado por el context window del modelo

---

## 💻 Ejercicios Completados

### ✅ Ejercicio Previo: API del Tiempo

**Objetivo**: Entender cómo funcionan las APIs con un ejemplo cotidiano.

**Lo que hiciste**:
- Construir una URL con parámetros
- Hacer una petición GET
- Parsear JSON
- Extraer y mostrar datos

**Concepto clave**: Todas las APIs funcionan así, incluidas las de IA.

### ✅ Ejercicio 1: Primera Llamada a IA Generativa

**Objetivo**: Hacer tu primera petición a Gemini y entender la estructura de la respuesta.

**Lo que hiciste**:
1. Configurar API key de forma segura
2. Preparar payload con system prompt y user message
3. Ver la respuesta en formato RAW y JSON
4. Parsear el JSON para extraer el texto generado
5. Entender el concepto de temperatura

**Concepto clave**: Las APIs de IA devuelven JSON estructurado que necesitas parsear.

**Concepto clave**: Puedes controlar la aleatoriedad/creatividad ajustando parámetros.

### ✅ Ejercicio 2: Conversación Multi-Turno con Memoria

**Objetivo**: Crear una conversación coherente donde la IA "recuerda" mensajes anteriores.

**Lo que hiciste**:
1. Inicializar un historial vacío
2. Crear función auxiliar que actualiza el historial automáticamente
3. Hacer 3 turnos de conversación con preguntas contextuales
4. Visualizar el historial completo
5. Entender cómo funciona la "memoria" en chatbots

**Concepto clave**: La IA no recuerda - tú le envías el contexto completo cada vez.

---

## 🎯 Conceptos Clave para Recordar

1. 🔄 **Las APIs de IA son stateless** - no recuerdan nada entre llamadas
2. 📦 **Siempre envías el historial completo** para simular memoria
3. 🌡️ **Temperature controla creatividad** - bajo = determinista, alto = creativo
4. 🎭 **System prompts definen comportamiento** - personalidad y estilo
5. 💬 **User prompts son las preguntas** - lo que quieres que haga
6. 📊 **JSON es el formato estándar** - aprende a navegar estructuras
7. 🔢 **Tokens = dinero** - cuanto más largo el contexto, más cuesta
8. 🎲 **candidateCount genera alternativas** - útil para comparar opciones

---

## 📖 Recursos Adicionales

### 🔗 Documentación Oficial
- **Gemini API**: https://ai.google.dev/docs
- **OpenAI API**: https://platform.openai.com/docs
- **Anthropic API**: https://docs.anthropic.com

### 📚 Aprender Más
- **Prompt Engineering Guide**: https://www.promptingguide.ai
- **Hugging Face**: https://huggingface.co/learn (cursos gratis)
- **DeepLearning.AI**: https://www.deeplearning.ai/short-courses/ (cursos con Andrew Ng)

### 🎮 Experimentar
- **Ollama**: https://ollama.ai (modelos locales gratis)
- **LM Studio**: https://lmstudio.ai (GUI para modelos locales)
- **Google AI Studio**: https://makersuite.google.com (playground de Gemini)

---


**🚀 ¡Mucha suerte en tu viaje con IA Generativa!**